# Person 2 — SegFormer Attention Hooks, Drift Figures & Loss Integration

**Role:** Transformer Lead (Kalana / Member 2)  
**Proposal:** *Explainability-Guided SegFormer for Forest Cover Segmentation*

This notebook continues from `segformer_baseline_colab.ipynb` (vanilla SegFormer-B0,
test Dice ≈ 0.87 / IoU ≈ 0.78). Here we complete the **extended** Person 2 work:

| # | Task | Proposal weeks | Status in this notebook |
|---|------|----------------|-------------------------|
| 1 | Attention-extraction hooks + adapted Grad-Rollout (§3.2) | W3–4 | Implemented + smoke-tested |
| 2 | Qualitative attention-drift figures (paper motivation) | W3–4 | Implemented |
| 3 | First integration pass with Person 3's Attention Consistency Loss | W3–4 | Short preliminary training |
| 4 | Handoff package for Person 4 (checkpoints + attention API) | W3–4 / W5–6 | Packaged under `checkpoints/` + `results/` |

**How to read this notebook:** every section starts with a markdown brief, then
heavily commented code. Smoke-scale defaults keep CPU runs feasible; bump the
config flags for Colab/GPU full-scale.

## 0 — Environment & config

Works locally (CPU smoke) or on Google Colab (GPU). Paths auto-detect.

In [1]:
# =============================================================================
# STEP 0: installs + imports
# -----------------------------------------------------------------------------
# transformers must use attn_implementation="eager" later — SDPA does not
# expose attention probability tensors, which Grad-Rollout needs.
# =============================================================================
import sys
import subprocess

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import transformers  # noqa: F401
except ImportError:
    _pip("transformers", "accelerate")

import os
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")  # safe for headless / Colab savefig
import matplotlib.pyplot as plt
from transformers import SegformerConfig, SegformerForSemanticSegmentation

print(f"torch={torch.__version__}  device={'cuda' if torch.cuda.is_available() else 'cpu'}")

torch=2.11.0+cu128  device=cuda


In [3]:
# =============================================================================
# STEP 0b: paths + smoke-scale knobs
# -----------------------------------------------------------------------------
# PERSON2_DIR  = this folder (Kalana-Person2)
# IMAGES/MASKS = full 5,108-pair Forest Segmented dataset already checked in
# SMOKE_*      = deliberately small so Task 3 finishes on CPU; set SMOKE=False
#                (and raise EPOCHS) when you have a Colab GPU.
# =============================================================================
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Adjust if your Drive layout differs:
    DATA_ROOT = Path("/content/drive/MyDrive/Kalana")
    PERSON2_DIR = Path("/content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2")
    PERSON2_DIR.mkdir(parents=True, exist_ok=True)
else:
    PERSON2_DIR = Path.cwd()
    # Allow running from repo root OR from inside Kalana-Person2/
    if (PERSON2_DIR / "images").is_dir():
        DATA_ROOT = PERSON2_DIR
    elif (PERSON2_DIR / "Phase1" / "Kalana-Person2" / "images").is_dir():
        PERSON2_DIR = PERSON2_DIR / "Phase1" / "Kalana-Person2"
        DATA_ROOT = PERSON2_DIR
    else:
        # notebook kernel started elsewhere — resolve relative to this file's folder idea
        PERSON2_DIR = Path(r"Phase1/Kalana-Person2").resolve()
        DATA_ROOT = PERSON2_DIR

IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
CKPT_DIR = PERSON2_DIR / "checkpoints"
RESULTS_DIR = PERSON2_DIR / "results"
FIG_DIR = RESULTS_DIR / "attention_drift_figures"
ATTN_DUMP_DIR = RESULTS_DIR / "attention_maps"
for d in (CKPT_DIR, RESULTS_DIR, FIG_DIR, ATTN_DUMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- experiment scale ---
SMOKE = True          # True = CPU-friendly subset; False = nearer to proposal §4.1
IMG_SIZE = 256
SEED = 42
LR = 6e-5
PRETRAINED_ENCODER = "nvidia/mit-b0"

if SMOKE:
    # Small on purpose: Grad-Rollout is B=1 and uses double backprop on CPU.
    N_TRAIN, N_VAL, N_TEST = 12, 4, 4
    EPOCHS_VANILLA = 2
    EPOCHS_ATT = 1
    BATCH_SIZE = 4
else:
    N_TRAIN, N_VAL, N_TEST = 3500, 750, 750
    EPOCHS_VANILLA = 10
    EPOCHS_ATT = 5
    BATCH_SIZE = 8

# Proposal §3.3 defaults (Person 3 tunes λ2 in Phase 2)
LAMBDA1 = 1.0
LAMBDA2 = 0.3
ATT_MODE = "mse"      # "mse" or "kl"
GAUSS_SIGMA = 8.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DATA_ROOT   = {DATA_ROOT}")
print(f"CKPT_DIR    = {CKPT_DIR}")
print(f"SMOKE={SMOKE}  N_TRAIN/VAL/TEST={N_TRAIN}/{N_VAL}/{N_TEST}  device={device}")
assert IMAGES_DIR.is_dir() and MASKS_DIR.is_dir(), f"Missing images/masks under {DATA_ROOT}"

Mounted at /content/drive
DATA_ROOT   = /content/drive/MyDrive/Kalana
CKPT_DIR    = /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/checkpoints
SMOKE=True  N_TRAIN/VAL/TEST=12/4/4  device=cuda


In [4]:
# =============================================================================
# STEP 0c: shared data helpers (same pairing convention as the baseline notebook)
# =============================================================================
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def mask_name_to_image_name(mask_filename: str) -> str:
    """'855_mask_01.jpg' -> '855_sat_01.jpg'."""
    return mask_filename.replace("_mask", "_sat")

def list_mask_files():
    return sorted(
        f for f in os.listdir(MASKS_DIR)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )

class ForestSegDataset(Dataset):
    """Returns (image CHW float ImageNet-norm, mask HW long {0,1})."""

    def __init__(self, mask_filenames, img_size=IMG_SIZE):
        self.mask_filenames = list(mask_filenames)
        self.img_size = img_size

    def __len__(self):
        return len(self.mask_filenames)

    def __getitem__(self, idx):
        mask_fname = self.mask_filenames[idx]
        image_fname = mask_name_to_image_name(mask_fname)

        image = Image.open(IMAGES_DIR / image_fname).convert("RGB")
        image = image.resize((self.img_size, self.img_size))
        mask = Image.open(MASKS_DIR / mask_fname).convert("L")
        mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        image = np.array(image, dtype=np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = torch.from_numpy(image).permute(2, 0, 1).float()

        mask = np.array(mask, dtype=np.int64)
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask).long()
        return image, mask

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_smoke_splits(n_train, n_val, n_test, seed=SEED):
    """Deterministic subset of the full mask list (not a replacement for the
    full 70/15/15 paper split — only for doable local runs)."""
    all_files = list_mask_files()
    rng = random.Random(seed)
    rng.shuffle(all_files)
    total = n_train + n_val + n_test
    subset = all_files[:total]
    train = subset[:n_train]
    val = subset[n_train:n_train + n_val]
    test = subset[n_train + n_val:]
    return train, val, test

set_seed()
train_files, val_files, test_files = make_smoke_splits(N_TRAIN, N_VAL, N_TEST)
print(f"paired masks available: {len(list_mask_files())}")
print(f"using train/val/test = {len(train_files)}/{len(val_files)}/{len(test_files)}")

paired masks available: 5108
using train/val/test = 12/4/4


---
## TASK 1 — Attention-extraction hooks + adapted Grad-Rollout (proposal §3.2)

### Why this exists
Vanilla SegFormer attention is usually only visualized *after* training.
We need the **attention probability tensors during a forward pass** so we can:
1. draw motivation figures (Task 2), and
2. supervise them with Person 3's Attention Consistency Loss (Task 3).

### Why stage 4 only
MiT-B0 stages 1–3 use **spatial-reduction attention** (`sr_ratio` 8/4/2), so
attention matrices are **rectangular**. Standard Attention Rollout needs
**square** \(N \times N\) matrices over a shared token grid. Stage 4 has
`sr_ratio=1` → square \(64 \times 64\) attention for a 256×256 input — the
only stage where Grad-Rollout is mathematically valid without further hacks.

### Deliverable
`AttentionExtractor` + `grad_rollout_attention_map` returning \(A \in [0,1]^{256\times256}\).

In [5]:
# =============================================================================
# TASK 1a: build SegFormer-B0 with eager attention
# -----------------------------------------------------------------------------
# CRITICAL: attn_implementation="eager"
#   The default "sdpa" backend never materializes attention_probs, so hooks
#   would see None and Grad-Rollout could not run.
# =============================================================================
MIT_B0_STAGE_CONFIG = dict(
    num_channels=3,
    num_encoder_blocks=4,
    depths=[2, 2, 2, 2],
    sr_ratios=[8, 4, 2, 1],
    hidden_sizes=[32, 64, 160, 256],
    num_attention_heads=[1, 2, 5, 8],
    patch_sizes=[7, 3, 3, 3],
    strides=[4, 2, 2, 2],
    mlp_ratios=[4, 4, 4, 4],
    decoder_hidden_size=256,
)

def build_segformer(num_labels=2, pretrained=True, output_attentions=True):
    """Same architecture as segformer_baseline_colab.ipynb, but eager attn."""
    if pretrained:
        model = SegformerForSemanticSegmentation.from_pretrained(
            PRETRAINED_ENCODER,
            num_labels=num_labels,
            id2label={0: "non_forest", 1: "forest"},
            label2id={"non_forest": 0, "forest": 1},
            ignore_mismatched_sizes=True,
            attn_implementation="eager",
        )
    else:
        cfg = SegformerConfig(
            num_labels=num_labels,
            attn_implementation="eager",
            **MIT_B0_STAGE_CONFIG,
        )
        model = SegformerForSemanticSegmentation(cfg)
    model.config.output_attentions = output_attentions
    return model.to(device)

def forest_prob(logits: torch.Tensor) -> torch.Tensor:
    """Softmax channel 1 = P(forest). logits: (B, C, H, W) -> (B, H, W)."""
    return F.softmax(logits, dim=1)[:, 1]

print("build_segformer / forest_prob ready")

build_segformer / forest_prob ready


In [6]:
# =============================================================================
# TASK 1b: AttentionExtractor — forward hooks on stage-4 attention modules
# -----------------------------------------------------------------------------
# SegFormer module path pattern (HuggingFace transformers):
#   model.segformer.stages.{stage0..3}.blocks.{b}.attention
# We register a forward hook on each stage-4 block's attention submodule.
# When output_attentions=True, attention.forward returns
#   (context, attention_probs) with attention_probs shape
#   (B, heads=8, N=64, N=64) for 256x256 inputs.
# =============================================================================
class AttentionExtractor:
    """Captures stage-4 (sr_ratio=1) attention maps via forward hooks."""

    def __init__(self, model: nn.Module, stage_index: int = 4):
        depths = model.config.depths
        if not (1 <= stage_index <= len(depths)):
            raise ValueError(f"stage_index must be in [1, {len(depths)}]")
        self.model = model
        self.stage_index = stage_index
        self.n_blocks = depths[stage_index - 1]

        self._modules = []
        named = dict(model.named_modules())
        for block_idx in range(self.n_blocks):
            # stage_index is 1-based in the paper; HF uses 0-based stage list
            name = f"segformer.stages.{stage_index - 1}.blocks.{block_idx}.attention"
            module = named.get(name)
            if module is None:
                raise RuntimeError(
                    f"Missing submodule '{name}'. transformers version mismatch?"
                )
            self._modules.append(module)

        self._captured = [None] * self.n_blocks
        self._handles = []

    def _make_hook(self, slot: int):
        def _hook(module, inputs, output):
            # Expect (context, attention_probs) when output_attentions=True
            attn_probs = output[1] if isinstance(output, tuple) and len(output) > 1 else None
            if attn_probs is None:
                raise RuntimeError(
                    "Hook fired without attention_probs — "
                    "call model(..., output_attentions=True) with eager attn."
                )
            self._captured[slot] = attn_probs
        return _hook

    def __enter__(self):
        self._handles = [
            mod.register_forward_hook(self._make_hook(i))
            for i, mod in enumerate(self._modules)
        ]
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self._handles:
            h.remove()
        self._handles = []

    def forward_with_attention(self, pixel_values: torch.Tensor, retain_grad: bool = False):
        """One forward pass → (outputs, list of stage-4 attention tensors)."""
        self._captured = [None] * self.n_blocks
        with self:
            outputs = self.model(pixel_values=pixel_values, output_attentions=True)
        if any(a is None for a in self._captured):
            raise RuntimeError("Not all stage-4 attention hooks fired.")
        if retain_grad:
            for a in self._captured:
                a.retain_grad()
        return outputs, list(self._captured)

print("AttentionExtractor defined")

AttentionExtractor defined


In [7]:
# =============================================================================
# TASK 1c: Adapted Gradient-weighted Attention Rollout (Chefer et al. + §3.2)
# -----------------------------------------------------------------------------
# Algorithm (per sample, batch size MUST be 1):
#   1. Forward with hooks; get stage-4 attentions A_l
#   2. Pick a scalar target (default: sum of predicted forest probability)
#   3. Backprop to get d(target)/d(A_l)
#   4. Grad-weight: ReLU(grad * attn), mean over heads
#   5. Rollout recursion with residual + row-normalize across stage-4 blocks
#   6. Row-mean relevance (no CLS token in SegFormer) → 8x8 grid
#   7. Bilinear upsample to 256x256, min-max normalize → A in [0, 1]
#
# create_graph=True  → training path (L_att needs double backprop)
# create_graph=False → cheap inference / figure path
# =============================================================================
def grad_rollout_attention_map(
    model: nn.Module,
    pixel_values: torch.Tensor,
    extractor=None,
    out_size=(IMG_SIZE, IMG_SIZE),
    create_graph: bool = False,
):
    if pixel_values.shape[0] != 1:
        raise ValueError("grad_rollout_attention_map expects batch size 1.")

    own = extractor is None
    if own:
        extractor = AttentionExtractor(model, stage_index=4)

    outputs, stage_attentions = extractor.forward_with_attention(
        pixel_values, retain_grad=not create_graph
    )

    # Target scalar: "what drove the model to predict forest?"
    logits_up = F.interpolate(
        outputs.logits, size=out_size, mode="bilinear", align_corners=False
    )
    target = forest_prob(logits_up).sum()

    if create_graph:
        grads = torch.autograd.grad(
            target, stage_attentions, create_graph=True, retain_graph=True
        )
    else:
        model.zero_grad(set_to_none=True)
        target.backward(retain_graph=False)
        grads = [attn.grad for attn in stage_attentions]
        if any(g is None for g in grads):
            raise RuntimeError("No gradient on stage-4 attention — check target_fn.")

    n_tokens = stage_attentions[0].shape[-1]  # 64 for 256 input
    R = torch.eye(n_tokens, device=stage_attentions[0].device)

    for attn, grad in zip(stage_attentions, grads):
        # Chefer: positive part of (grad ⊙ attn), average heads → (N, N)
        weighted = (grad * attn).clamp(min=0).mean(dim=1)[0]
        a_hat = weighted + torch.eye(n_tokens, device=weighted.device)
        a_hat = a_hat / a_hat.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        R = a_hat @ R

    relevance = R.mean(dim=0).clamp(min=0)  # row-mean (no CLS)
    rmax = relevance.max()
    relevance = relevance / rmax if rmax > 0 else relevance

    grid = int(round(n_tokens ** 0.5))  # 8
    attn_map = relevance.reshape(1, 1, grid, grid)
    attn_map = F.interpolate(attn_map, size=out_size, mode="bilinear", align_corners=False)
    attn_map = attn_map[0, 0]
    if not create_graph:
        attn_map = attn_map.detach()
    return attn_map, outputs

print("grad_rollout_attention_map defined")

grad_rollout_attention_map defined


In [8]:
# =============================================================================
# TASK 1d: smoke test — one image through hooks + Grad-Rollout
# -----------------------------------------------------------------------------
# Expected: attn_map shape (256, 256), values in [0, 1].
# We use pretrained=True once so encoder weights exist; decode head is random
# until Task 3 fine-tunes — that is fine for a shape/API smoke test.
# =============================================================================
print("TASK 1 smoke: loading SegFormer-B0 (pretrained MiT-B0 encoder)...")
model_smoke = build_segformer(pretrained=True)
model_smoke.eval()

ds_one = ForestSegDataset(train_files[:1])
x0, y0 = ds_one[0]
x0 = x0.unsqueeze(0).to(device)

with torch.enable_grad():
    attn0, out0 = grad_rollout_attention_map(model_smoke, x0, create_graph=False)

print(f"  logits shape     : {tuple(out0.logits.shape)}")
print(f"  attention map    : {tuple(attn0.shape)}  min={attn0.min():.4f} max={attn0.max():.4f}")
assert attn0.shape == (IMG_SIZE, IMG_SIZE)
assert 0.0 <= float(attn0.min()) and float(attn0.max()) <= 1.0 + 1e-5
print("TASK 1 PASSED — hooks + Grad-Rollout produce a valid A map.")

# Free GPU/CPU memory before later tasks (we rebuild as needed)
del model_smoke, out0
torch.cuda.empty_cache() if torch.cuda.is_available() else None

TASK 1 smoke: loading SegFormer-B0 (pretrained MiT-B0 encoder)...


config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.3MB            

model.safetensors: downloading bytes:           |  0.00B            

  logits shape     : (1, 2, 64, 64)
  attention map    : (256, 256)  min=0.0437 max=1.0000
TASK 1 PASSED — hooks + Grad-Rollout produce a valid A map.


---
## TASK 2 — Qualitative attention-drift figures

### Goal
Produce **2–3 side-by-side figures** for the short paper motivation section:

`RGB image | GT forest mask | Grad-Rollout attention overlay`

These show where vanilla SegFormer attention lands (often roads / shadows /
non-forest structure) versus the canopy. After Task 3 trains an
attention-consistency checkpoint, we also compare vanilla vs +L_att columns.

Figures are written to `results/attention_drift_figures/`.

In [9]:
# =============================================================================
# TASK 2a: figure helpers
# =============================================================================
def attention_overlay(image_01: np.ndarray, attn_map: np.ndarray) -> np.ndarray:
    """Blend RGB image (H,W,3 in [0,1]) with a jet heatmap of attention."""
    heat = plt.get_cmap("jet")(np.clip(attn_map, 0, 1))[..., :3]
    return np.clip(0.55 * image_01 + 0.45 * heat, 0, 1)

def load_rgb_01(mask_fname: str) -> np.ndarray:
    img = Image.open(IMAGES_DIR / mask_name_to_image_name(mask_fname)).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))
    return np.array(img, dtype=np.float32) / 255.0

def load_mask_01(mask_fname: str) -> np.ndarray:
    m = Image.open(MASKS_DIR / mask_fname).convert("L")
    m = m.resize((IMG_SIZE, IMG_SIZE), resample=Image.NEAREST)
    return (np.array(m) > 127).astype(np.float32)

def load_ckpt_model(path: Path) -> nn.Module:
    """Load a checkpoint saved by this notebook (or Person 3 smoke format)."""
    model = build_segformer(pretrained=False)
    ckpt = torch.load(path, map_location=device)
    state = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model

def save_drift_figure(mask_fname, attn_vanilla, attn_att=None, tag="demo"):
    rgb = load_rgb_01(mask_fname)
    gt = load_mask_01(mask_fname)
    n_cols = 4 if attn_att is not None else 3
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    axes[0].imshow(rgb); axes[0].set_title("Image")
    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1); axes[1].set_title("GT forest mask")
    axes[2].imshow(attention_overlay(rgb, attn_vanilla))
    axes[2].set_title("Vanilla Grad-Rollout")
    if attn_att is not None:
        axes[3].imshow(attention_overlay(rgb, attn_att))
        axes[3].set_title("+ Att. Consistency")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    out = FIG_DIR / f"attention_drift_{tag}.png"
    fig.savefig(out, dpi=130)
    plt.close(fig)
    print(f"  wrote {out}")
    return out

print(f"Figure output dir: {FIG_DIR}")

Figure output dir: /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures


In [10]:
# =============================================================================
# TASK 2b: generate motivation figures from the best available vanilla weights
# -----------------------------------------------------------------------------
# Preference order:
#   1) checkpoints/segformer_b0_vanilla_best.pt  (produced later by Task 3)
#   2) any existing baseline .pt the user drops into checkpoints/
#   3) freshly built pretrained encoder (decode head random) — still useful
#      to prove the figure pipeline; replace after Task 3.
# =============================================================================
VANILLA_CKPT = CKPT_DIR / "segformer_b0_vanilla_best.pt"
ATT_CKPT = CKPT_DIR / "segformer_b0_att_best.pt"
BASELINE_FALLBACK = CKPT_DIR / "segformer_b0_baseline.pt"  # optional Drop from Colab

def resolve_vanilla_model():
    for p in (VANILLA_CKPT, BASELINE_FALLBACK):
        if p.exists():
            print(f"Loading vanilla weights from {p.name}")
            return load_ckpt_model(p), p.name
    print("No vanilla checkpoint yet — using pretrained encoder (figures are pipeline demos).")
    m = build_segformer(pretrained=True)
    m.eval()
    return m, "pretrained_encoder_only"

model_v, v_tag = resolve_vanilla_model()
model_a = load_ckpt_model(ATT_CKPT) if ATT_CKPT.exists() else None
if model_a is not None:
    print(f"Also loaded attention-consistency checkpoint: {ATT_CKPT.name}")

# Pick 3 held-out-ish demo images from the test split
demo_files = test_files[:3] if len(test_files) >= 3 else train_files[:3]
figure_paths = []
for i, mf in enumerate(demo_files, start=1):
    x, _ = ForestSegDataset([mf])[0]
    x = x.unsqueeze(0).to(device)
    with torch.enable_grad():
        a_v, _ = grad_rollout_attention_map(model_v, x, create_graph=False)
        a_a = None
        if model_a is not None:
            a_a, _ = grad_rollout_attention_map(model_a, x, create_graph=False)
    figure_paths.append(
        save_drift_figure(
            mf,
            a_v.detach().cpu().numpy(),
            None if a_a is None else a_a.detach().cpu().numpy(),
            tag=f"{i:02d}_{v_tag}",
        )
    )

print(f"TASK 2 DONE — {len(figure_paths)} figure(s) in {FIG_DIR}")
# Keep models only if Task 3 needs them; free for now
del model_v
if model_a is not None:
    del model_a
torch.cuda.empty_cache() if torch.cuda.is_available() else None

No vanilla checkpoint yet — using pretrained encoder (figures are pipeline demos).


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   

  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_01_pretrained_encoder_only.png
  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_02_pretrained_encoder_only.png
  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_03_pretrained_encoder_only.png
TASK 2 DONE — 3 figure(s) in /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures


---
## TASK 3 — Integrate Person 3's Attention Consistency Loss + preliminary train

### Math (proposal §3.3)
\[
A^\* = \mathrm{Gaussian}(Y),\quad
L_\mathrm{att}=\mathrm{MSE}(A,A^\*)\ \mathrm{or}\ \mathrm{KL}(A\|A^\*)
\]
\[
L = L_\mathrm{dice} + \lambda_1 L_\mathrm{bce} + \lambda_2 L_\mathrm{att}
\quad(\lambda_1=1,\ \lambda_2=0.3)
\]

`A` comes from **our** Grad-Rollout (`create_graph=True` so `L_att` can
backprop into the encoder). This is the Person 2 ↔ Person 3 integration pass.

We train two short runs:
1. **vanilla** — Dice+BCE only (reproducible local vanilla checkpoint)
2. **att** — full objective with \(L_\mathrm{att}\)

Checkpoints land in `Kalana-Person2/checkpoints/`.

In [11]:
# =============================================================================
# TASK 3a: Person 3 loss pieces (copied here so this notebook is self-contained)
# -----------------------------------------------------------------------------
# Canonical unit-tested implementation also lives in:
#   Phase1/Dinura-Person3/attention_consistency/loss.py
# Keep formulas identical so Person 4 evaluates one consistent definition.
# =============================================================================
def _gaussian_kernel1d(sigma, radius, device=None, dtype=None):
    x = torch.arange(-radius, radius + 1, device=device, dtype=dtype or torch.float32)
    kernel = torch.exp(-(x ** 2) / (2 * sigma ** 2))
    return kernel / kernel.sum()

def gaussian_soft_target(mask: torch.Tensor, sigma: float = GAUSS_SIGMA) -> torch.Tensor:
    """A* = Gaussian(Y). mask: (B,H,W) or (H,W) binary → soft target same shape."""
    if mask.dim() == 2:
        mask = mask.unsqueeze(0)
    mask = mask.float().unsqueeze(1)  # (B,1,H,W)
    radius = max(1, int(round(3 * sigma)))
    k = _gaussian_kernel1d(sigma, radius, device=mask.device, dtype=mask.dtype)
    kx, ky = k.view(1, 1, 1, -1), k.view(1, 1, -1, 1)
    smoothed = F.conv2d(mask, kx, padding=(0, radius))
    smoothed = F.conv2d(smoothed, ky, padding=(radius, 0))
    flat = smoothed.view(smoothed.shape[0], -1)
    vmax = flat.max(dim=1, keepdim=True).values.clamp_min(1e-8)
    smoothed = (flat / vmax).view_as(smoothed)
    return smoothed.squeeze(1)

class AttentionConsistencyLoss(nn.Module):
    """L_att = MSE(A, A*) or KL(A || A*)."""

    def __init__(self, mode=ATT_MODE, sigma=GAUSS_SIGMA, eps=1e-8):
        super().__init__()
        assert mode in ("mse", "kl")
        self.mode, self.sigma, self.eps = mode, sigma, eps

    def forward(self, attention, mask):
        if attention.dim() == 2:
            attention = attention.unsqueeze(0)
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        target = gaussian_soft_target(mask, sigma=self.sigma).to(attention.dtype)
        if self.mode == "mse":
            return F.mse_loss(attention, target)
        b = attention.shape[0]
        a = attention.reshape(b, -1).clamp_min(self.eps)
        t = target.reshape(b, -1).clamp_min(self.eps)
        a = a / a.sum(dim=1, keepdim=True)
        t = t / t.sum(dim=1, keepdim=True)
        return (a * (a.log() - t.log())).sum(dim=1).mean()

def dice_bce(probs, target, eps=1e-6):
    """probs/target: (B,H,W) float. Returns (L_dice, L_bce)."""
    target = target.float()
    inter = (probs * target).sum(dim=(1, 2))
    union = probs.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    l_dice = (1 - (2 * inter + eps) / (union + eps)).mean()
    l_bce = F.binary_cross_entropy(probs.clamp(eps, 1 - eps), target, reduction="mean")
    return l_dice, l_bce

@torch.no_grad()
def batch_dice_iou(probs, target, thr=0.5, eps=1e-6):
    pred = (probs > thr).float()
    target = target.float()
    inter = (pred * target).sum(dim=(1, 2))
    union = pred.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
    dice = ((2 * inter + eps) / (union + eps)).mean().item()
    iou = ((inter + eps) / (union - inter + eps)).mean().item()
    return dice, iou

print("AttentionConsistencyLoss + dice/bce metrics ready")

AttentionConsistencyLoss + dice/bce metrics ready


In [12]:
# =============================================================================
# TASK 3b: training loops
# -----------------------------------------------------------------------------
# vanilla : normal batched train (Dice + BCE)
# att     : batch size 1 required (Grad-Rollout constraint) + create_graph=True
# =============================================================================
def run_epoch_vanilla(model, loader, opt=None):
    is_train = opt is not None
    model.train(is_train)
    tot_loss = tot_dice = tot_iou = 0.0
    n = 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train:
                opt.zero_grad()
            out = model(pixel_values=x)
            logits = F.interpolate(out.logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            probs = forest_prob(logits)
            l_dice, l_bce = dice_bce(probs, y)
            loss = l_dice + LAMBDA1 * l_bce
            if is_train:
                loss.backward()
                opt.step()
            d, iou = batch_dice_iou(probs.detach(), y)
            tot_loss += loss.item(); tot_dice += d; tot_iou += iou; n += 1
    return tot_loss / n, tot_dice / n, tot_iou / n

def run_epoch_attention(model, dataset, opt, att_loss_fn, is_train):
    """Per-image loop (B=1) with Grad-Rollout inside the graph when training."""
    model.train()  # need live graph even for "val" when measuring L_att
    tot_loss = tot_dice = tot_iou = tot_att = 0.0
    n = len(dataset)
    for i in range(n):
        x, y = dataset[i]
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)
        if is_train:
            opt.zero_grad()
        attn_map, outputs = grad_rollout_attention_map(
            model, x, create_graph=is_train
        )
        logits = F.interpolate(
            outputs.logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False
        )
        probs = forest_prob(logits)
        l_dice, l_bce = dice_bce(probs, y)
        l_att = att_loss_fn(attn_map, y[0].float())
        loss = l_dice + LAMBDA1 * l_bce + LAMBDA2 * l_att
        if is_train:
            loss.backward()
            opt.step()
        d, iou = batch_dice_iou(probs.detach(), y)
        tot_loss += loss.item(); tot_dice += d; tot_iou += iou; tot_att += l_att.item()
    return tot_loss / n, tot_dice / n, tot_iou / n, tot_att / n

def train_variant(variant: str, epochs: int):
    print(f"\n===== Training variant: {variant} ({epochs} epochs) =====")
    train_ds = ForestSegDataset(train_files)
    val_ds = ForestSegDataset(val_files)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = build_segformer(pretrained=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    att_fn = AttentionConsistencyLoss(mode=ATT_MODE, sigma=GAUSS_SIGMA)

    best_dice = -1.0
    history = []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        if variant == "vanilla":
            tr = run_epoch_vanilla(model, train_loader, opt)
            va = run_epoch_vanilla(model, val_loader, opt=None)
            row = dict(epoch=epoch, train_loss=tr[0], train_dice=tr[1], train_iou=tr[2],
                       val_loss=va[0], val_dice=va[1], val_iou=va[2])
        else:
            tr = run_epoch_attention(model, train_ds, opt, att_fn, is_train=True)
            va = run_epoch_attention(model, val_ds, opt, att_fn, is_train=False)
            row = dict(epoch=epoch, train_loss=tr[0], train_dice=tr[1], train_iou=tr[2],
                       val_loss=va[0], val_dice=va[1], val_iou=va[2],
                       train_l_att=tr[3], val_l_att=va[3])
        history.append(row)
        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train_dice={row['train_dice']:.4f} val_dice={row['val_dice']:.4f} "
            f"val_iou={row['val_iou']:.4f}"
            + (f" val_Latt={row['val_l_att']:.4f}" if "val_l_att" in row else "")
        )
        if row["val_dice"] > best_dice:
            best_dice = row["val_dice"]
            out = CKPT_DIR / f"segformer_b0_{variant}_best.pt"
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "val_dice": row["val_dice"],
                    "val_iou": row["val_iou"],
                    "variant": variant,
                    "seed": SEED,
                    "n_train": len(train_files),
                    "n_val": len(val_files),
                },
                out,
            )
            print(f"  -> saved {out.name}")

    last = CKPT_DIR / f"segformer_b0_{variant}_last.pt"
    torch.save(
        {"model_state": model.state_dict(), "epoch": epochs, "variant": variant,
         "val_dice": history[-1]["val_dice"], "val_iou": history[-1]["val_iou"]},
        last,
    )
    summary = {
        "variant": variant,
        "epochs": epochs,
        "best_val_dice": best_dice,
        "final_val_dice": history[-1]["val_dice"],
        "final_val_iou": history[-1]["val_iou"],
        "wall_clock_s": round(time.time() - t0, 1),
        "smoke": SMOKE,
        "lambda2": LAMBDA2 if variant == "att" else None,
        "att_mode": ATT_MODE if variant == "att" else None,
    }
    with open(RESULTS_DIR / f"train_summary_{variant}.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(f"TASK 3/{variant} done in {summary['wall_clock_s']}s — best val dice={best_dice:.4f}")
    return summary

# Run both variants (Person 2 integration deliverable)
summary_vanilla = train_variant("vanilla", EPOCHS_VANILLA)
summary_att = train_variant("att", EPOCHS_ATT)
print("TASK 3 PASSED — preliminary vanilla + attention-consistency checkpoints written.")


===== Training variant: vanilla (2 epochs) =====


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   

epoch 01/2 | train_dice=0.4456 val_dice=0.6828 val_iou=0.5474
  -> saved segformer_b0_vanilla_best.pt
epoch 02/2 | train_dice=0.7250 val_dice=0.7501 val_iou=0.6243
  -> saved segformer_b0_vanilla_best.pt
TASK 3/vanilla done in 30.2s — best val dice=0.7501

===== Training variant: att (1 epochs) =====


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   

epoch 01/1 | train_dice=0.5652 val_dice=0.5988 val_iou=0.4415 val_Latt=0.4595
  -> saved segformer_b0_att_best.pt
TASK 3/att done in 1.8s — best val dice=0.5988
TASK 3 PASSED — preliminary vanilla + attention-consistency checkpoints written.


In [13]:
# =============================================================================
# TASK 3c: refresh drift figures now that real checkpoints exist
# =============================================================================
model_v = load_ckpt_model(VANILLA_CKPT)
model_a = load_ckpt_model(ATT_CKPT)
for i, mf in enumerate(demo_files, start=1):
    x, _ = ForestSegDataset([mf])[0]
    x = x.unsqueeze(0).to(device)
    with torch.enable_grad():
        a_v, _ = grad_rollout_attention_map(model_v, x, create_graph=False)
        a_a, _ = grad_rollout_attention_map(model_a, x, create_graph=False)
    save_drift_figure(
        mf,
        a_v.detach().cpu().numpy(),
        a_a.detach().cpu().numpy(),
        tag=f"{i:02d}_post_train",
    )
print("Post-train attention-drift figures updated.")
del model_v, model_a

  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_01_post_train.png
  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_02_post_train.png
  wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_drift_figures/attention_drift_03_post_train.png
Post-train attention-drift figures updated.


---
## TASK 4 — Handoff package for Person 4 (Evaluation Lead)

Person 4's evaluator expects:
- `segformer_b0_vanilla.pt` / `segformer_b0_att_consistency.pt` under their
  `Lasana-Person4_Evaluation/checkpoints/` (see their `config.py`)
- an attention-map API: given `(model, pixel_values)` → `(H,W)` map in `[0,1]`

This section:
1. Copies our best checkpoints into Person 4's folder (and keeps local copies)
2. Dumps example attention `.npy` maps for AAMO smoke tests
3. Writes `results/PERSON4_HANDOFF.md` describing the API

In [14]:
# =============================================================================
# TASK 4a: copy checkpoints to Person 4's expected filenames
# =============================================================================
PERSON4_CKPT_DIR = PERSON2_DIR.parent / "Lasana-Person4_Evaluation" / "checkpoints"
PERSON4_CKPT_DIR.mkdir(parents=True, exist_ok=True)

handoff_copies = {
    VANILLA_CKPT: [
        CKPT_DIR / "segformer_b0_vanilla.pt",
        PERSON4_CKPT_DIR / "segformer_b0_vanilla.pt",
    ],
    ATT_CKPT: [
        CKPT_DIR / "segformer_b0_att_consistency.pt",
        PERSON4_CKPT_DIR / "segformer_b0_att_consistency.pt",
    ],
}

import shutil
copied = []
for src, dests in handoff_copies.items():
    if not src.exists():
        print(f"WARNING: missing {src.name} — run Task 3 first")
        continue
    for dst in dests:
        shutil.copy2(src, dst)
        copied.append(str(dst))
        print(f"copied {src.name} -> {dst}")

print(f"Copied {len(copied)} checkpoint file(s).")

copied segformer_b0_vanilla_best.pt -> /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/checkpoints/segformer_b0_vanilla.pt
copied segformer_b0_vanilla_best.pt -> /content/drive/MyDrive/DNN_Project/Phase1/Lasana-Person4_Evaluation/checkpoints/segformer_b0_vanilla.pt
copied segformer_b0_att_best.pt -> /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/checkpoints/segformer_b0_att_consistency.pt
copied segformer_b0_att_best.pt -> /content/drive/MyDrive/DNN_Project/Phase1/Lasana-Person4_Evaluation/checkpoints/segformer_b0_att_consistency.pt
Copied 4 checkpoint file(s).


In [15]:
# =============================================================================
# TASK 4b: dump sample attention maps (.npy) for Person 4 AAMO checks
# -----------------------------------------------------------------------------
# Format: float32 array shaped (H, W) in [0, 1], one file per image.
# Also save a tiny JSON index so evaluate.py --attention-npy users know which
# mask each map belongs to.
# =============================================================================
model_v = load_ckpt_model(VANILLA_CKPT)
model_a = load_ckpt_model(ATT_CKPT)
index = []
for i, mf in enumerate(demo_files):
    x, y = ForestSegDataset([mf])[0]
    x_b = x.unsqueeze(0).to(device)
    with torch.enable_grad():
        a_v, _ = grad_rollout_attention_map(model_v, x_b, create_graph=False)
        a_a, _ = grad_rollout_attention_map(model_a, x_b, create_graph=False)
    stem = Path(mf).stem
    p_v = ATTN_DUMP_DIR / f"{stem}_vanilla_attn.npy"
    p_a = ATTN_DUMP_DIR / f"{stem}_att_attn.npy"
    np.save(p_v, a_v.detach().cpu().numpy().astype(np.float32))
    np.save(p_a, a_a.detach().cpu().numpy().astype(np.float32))
    index.append({
        "mask_file": mf,
        "image_file": mask_name_to_image_name(mf),
        "vanilla_attn_npy": str(p_v.name),
        "att_attn_npy": str(p_a.name),
        "mask_forest_frac": float(y.float().mean()),
    })
    print(f"  dumped attention for {mf}")

with open(ATTN_DUMP_DIR / "index.json", "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2)

del model_v, model_a
print(f"Attention dumps in {ATTN_DUMP_DIR}")

  dumped attention for 877160_mask_71.jpg
  dumped attention for 992507_mask_38.jpg
  dumped attention for 406425_mask_65.jpg
Attention dumps in /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/attention_maps


In [16]:
# =============================================================================
# TASK 4c: write handoff README for Person 4 / future teammates
# =============================================================================
lines = [
    "# Person 2 -> Person 4 handoff",
    "",
    "Generated by `segformer_attention.ipynb` (Kalana / Member 2).",
    "",
    "## Checkpoints",
    "",
    "| Variant | Local path | Person 4 path (`config.py`) |",
    "|---------|------------|-----------------------------|",
    "| Vanilla SegFormer-B0 | `Kalana-Person2/checkpoints/segformer_b0_vanilla.pt` | `Lasana-Person4_Evaluation/checkpoints/segformer_b0_vanilla.pt` |",
    "| + Attention Consistency | `Kalana-Person2/checkpoints/segformer_b0_att_consistency.pt` | `Lasana-Person4_Evaluation/checkpoints/segformer_b0_att_consistency.pt` |",
    "",
    "Checkpoint dict keys: `model_state`, `epoch`, `val_dice`, `val_iou`, `variant`, `seed`.",
    "",
    "## Attention-map API (for AAMO)",
    "",
    "```python",
    "# Inside this notebook (or Dinura-Person3/attention_consistency):",
    "attn_map, outputs = grad_rollout_attention_map(model, pixel_values, create_graph=False)",
    "# attn_map: (256, 256) float tensor in [0, 1]; pixel_values batch size MUST be 1",
    "```",
    "",
    "Precomputed examples: `Kalana-Person2/results/attention_maps/` (see `index.json`).",
    "",
    "## Qualitative figures",
    "",
    "`Kalana-Person2/results/attention_drift_figures/attention_drift_*_post_train.png`",
    "",
    "## Smoke-run summaries",
    "",
    "- `results/train_summary_vanilla.json`",
    "- `results/train_summary_att.json`",
    "",
    "> If SMOKE=True, numbers are preliminary pipeline-proof, not paper-final.",
    "> Re-run with SMOKE=False on Colab GPU for proposal-scale results.",
    "",
]
out_md = RESULTS_DIR / "PERSON4_HANDOFF.md"
out_md.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {out_md}")

print("\n=== Person 2 deliverable inventory ===")
for p in sorted(CKPT_DIR.glob("*.pt")):
    print(f"  ckpt  {p.name:40s} {p.stat().st_size/1e6:.1f} MB")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  fig   {p.name}")
for p in sorted(ATTN_DUMP_DIR.glob("*")):
    print(f"  attn  {p.name}")
print("TASK 4 PASSED — Person 4 handoff package ready.")

Wrote /content/drive/MyDrive/DNN_Project/Phase1/Kalana-Person2/results/PERSON4_HANDOFF.md

=== Person 2 deliverable inventory ===
  ckpt  segformer_b0_att_best.pt                 14.9 MB
  ckpt  segformer_b0_att_consistency.pt          14.9 MB
  ckpt  segformer_b0_att_last.pt                 14.9 MB
  ckpt  segformer_b0_vanilla.pt                  14.9 MB
  ckpt  segformer_b0_vanilla_best.pt             14.9 MB
  ckpt  segformer_b0_vanilla_last.pt             14.9 MB
  fig   attention_drift_01_post_train.png
  fig   attention_drift_01_pretrained_encoder_only.png
  fig   attention_drift_02_post_train.png
  fig   attention_drift_02_pretrained_encoder_only.png
  fig   attention_drift_03_post_train.png
  fig   attention_drift_03_pretrained_encoder_only.png
  attn  406425_mask_65_att_attn.npy
  attn  406425_mask_65_vanilla_attn.npy
  attn  877160_mask_71_att_attn.npy
  attn  877160_mask_71_vanilla_attn.npy
  attn  992507_mask_38_att_attn.npy
  attn  992507_mask_38_vanilla_attn.npy
  attn  i

---
## Done — what you just built

1. **Hooks + Grad-Rollout** adapted to SegFormer stage-4 spatial-reduction constraints  
2. **Attention-drift figures** for the short-paper motivation  
3. **Preliminary explainability-guided checkpoint** via Person 3's \(L_\mathrm{att}\)  
4. **Handoff** checkpoints + attention API notes for Person 4  

**Next (Phase 2 support, Weeks 5–6):** help Person 3 with full-scale λ sweeps;
keep this attention API stable so Person 4's ablation rows stay reproducible.